In [1]:
import subprocess
import time
import itertools
import os
import sys
import pandas as pd
import glob
from datetime import datetime

# ================= 設定區 =================
dic = {
    'batch_size': [1000],
    'epoch': [500],
    'layer': [2],
    'hidden': [32],
    'data_version': [58],
    'lr': [0.003],
    'column': ['acceleration_X,acceleration_Y,acceleration_Z,gyroscope_X,gyroscope_Y,gyroscope_Z'],
    'folds': [1],
    # ==== 邊緣端校正與測試路徑 ====
    #'target_version': ['58/special_data_hengling_yi'],  # 使用者前5分鐘校正數據
    #'test_version': ['58/test_data_hengling_yi'],     # 實際測試推論數據
    #'target_version': ['58/special_data_hengling_yi'],  # 使用者前5分鐘校正數據
    #'test_version': ['58/test_data'],     # 實際測試推論數據
    'target_version': ['59/special_data_manabang'],  # 使用者前5分鐘校正數據
    'test_version': ['59/test_data_6'],     # 實際測試推論數據
    'lambda_dann': [1.0]                    # 保留相容槽
}

MAX_CONCURRENT_JOBS = 4  
TRAIN_SCRIPT = "train.py" 
TEST_SCRIPT = "test2.py"   
OUTPUT_LOG_DIR = "./test_log"
# =========================================

def get_combinations(params):
    keys = list(params.keys())
    values = list(params.values())
    for combo in itertools.product(*values):
        yield dict(zip(keys, combo))

def run_phase(phase_name, script_name, combinations, max_jobs):
    print(f"\n=== 開始執行階段: {phase_name} ===")
    total_jobs = len(combinations)
    running_processes = []
    
    for i, p in enumerate(combinations):
        cmd = [
            'python3', script_name,
            f'--batch_size={p["batch_size"]}',
            f'--epoch={p["epoch"]}',
            f'--layer={p["layer"]}',
            f'--hidden={p["hidden"]}',
            f'--data_version={p["data_version"]}',
            f'--lr={p["lr"]}',
            f'--column={p["column"]}',
            f'--fold={p["folds"]}', 
            f'--lambda_dann={p["lambda_dann"]}',
            f'--target_version={p["target_version"]}'
        ]
        
        # 若為測試階段，需額外傳入 test_version
        if phase_name == "Testing":
            cmd.append(f'--test_version={p["test_version"]}')
        
        proc = subprocess.Popen(cmd, stdout=subprocess.DEVNULL)
        #proc = subprocess.Popen(cmd)
        running_processes.append(proc)
        
        if i % 10 == 0:
            print(f"[{phase_name}] 進度: {i}/{total_jobs} (Running: {len(running_processes)})")

        while len(running_processes) >= max_jobs:
            running_processes = [proc for proc in running_processes if proc.poll() is None]
            if len(running_processes) >= max_jobs:
                time.sleep(1)

    for proc in running_processes:
        proc.wait()
    print(f"=== {phase_name} 階段完成 ===\n")

def collect_results():
    print(f"=== 正在從 {OUTPUT_LOG_DIR} 彙整最新測試報告 ===")
    
    # 修改搜尋關鍵字：適配新版測試程式的 EdgeAdapted 命名
    result_files = glob.glob(os.path.join(OUTPUT_LOG_DIR, "*_EdgeAdapted_result.csv"))
    
    if not result_files:
        result_files = glob.glob(os.path.join(OUTPUT_LOG_DIR, "*.csv"))
        if not result_files or any("Final_Report" in f for f in result_files):
            print(f"在 {OUTPUT_LOG_DIR} 找不到任何有效的測試結果檔案。")
            return

    all_dfs = []
    for f in result_files:
        if "Final_Report" in f: continue 
        try:
            df = pd.read_csv(f)
            if not df.empty:
                all_dfs.append(df)
        except Exception as e:
            print(f"讀取 {f} 失敗: {e}")
    
    if all_dfs:
        final_df = pd.concat(all_dfs, ignore_index=True)
        
        # 修改：適配新測試程式的消融實驗欄位前綴（以最優的特徵對齊+動態閾值完全體為排序基準）
        sort_col = "adapt_macro_f1"
        if sort_col in final_df.columns:
            final_df = final_df.sort_values(by=sort_col, ascending=False)
        elif "macro_f1" in final_df.columns:
            final_df = final_df.sort_values(by="macro_f1", ascending=False)
            
        out_name = f"Final_Report_EdgeAdapted_v{dic['data_version'][0]}.csv"
        final_df.to_csv(out_name, index=False)
        
        print("-" * 50)
        print(f"報告整合完成！共匯總 {len(all_dfs)} 筆測試數據。")
        print(f"最終報告已產出: {out_name}")
        print("-" * 50)
        print("Top 5 最佳模型表現 (Z-Score + 高維 GRU 特徵 CORAL 校正完全體表現):")
        
        # 修改：改為列印適配新測試程式欄位名（列出 BASE 基準與 ADAPT 完全體的對比）
        display_cols = [
            'run_name', 
            'base_acc', 'base_macro_f1', 
            'adapt_acc', 'adapt_macro_f1', 
            'adapt_f1_notTired', 'adapt_f1_Tired', 'adapt_f1_Other'
        ]
        
        try:
            # 確保要列印的欄位都在 final_df 中
            existing_cols = [c for c in display_cols if c in final_df.columns]
            print(final_df.head(5)[existing_cols])
        except Exception as e:
            print(final_df.head(5))
            
    else:
        print("沒有有效的數據可以合併。")

def main():
    start_time = datetime.now()
    combinations = list(get_combinations(dic)) 
    
    #run_phase("Training", TRAIN_SCRIPT, combinations, max_jobs=MAX_CONCURRENT_JOBS)
    
    # 提醒：如果您要進行推論與適配評估，記得解除下方 Testing 的註解
    run_phase("Testing", TEST_SCRIPT, combinations, max_jobs=MAX_CONCURRENT_JOBS)
    
    collect_results()

    end_time = datetime.now()
    print(f"總耗時: {end_time - start_time}")

if __name__ == "__main__":
    main()


=== 開始執行階段: Testing ===
[Testing] 進度: 0/1 (Running: 1)
=== Testing 階段完成 ===

=== 正在從 ./test_log 彙整最新測試報告 ===
--------------------------------------------------
報告整合完成！共匯總 1 筆測試數據。
最終報告已產出: Final_Report_EdgeAdapted_v58.csv
--------------------------------------------------
Top 5 最佳模型表現 (Z-Score + 高維 GRU 特徵 CORAL 校正完全體表現):
                                             run_name  base_acc  \
0   fold1_layer_2_hidden_32_lr_0.003_ServerMaster_...  0.962351   
1   fold1_layer_2_hidden_32_lr_0.003_ServerMaster_...  0.962351   
6   fold1_layer_2_hidden_32_lr_0.003_ServerMaster_...  0.962351   
10  fold1_layer_2_hidden_32_lr_0.003_ServerMaster_...  0.793388   
9   fold1_layer_2_hidden_32_lr_0.003_ServerMaster_...  0.793388   

    base_macro_f1  adapt_acc  adapt_macro_f1  adapt_f1_notTired  \
0        0.947529   0.887715        0.861048           0.880874   
1        0.947529   0.887715        0.861048           0.880874   
6        0.947529   0.887715        0.861048           0.880874   
10   